# **Loading Dataset For Analysis**

In [2]:
import pandas as pd

train_df = pd.read_csv('Dataset/train.csv',encoding='latin-1')
train_df

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26
...,...,...,...,...,...,...,...,...,...,...
27476,4eac33d1c0,wish we could come see u on Denver husband l...,d lost,negative,night,31-45,Ghana,31072940,227540.0,137
27477,4f4c4fc327,I`ve wondered about rake to. The client has ...,", don`t force",negative,morning,46-60,Greece,10423054,128900.0,81
27478,f67aae2310,Yay good for both of you. Enjoy the break - y...,Yay good for both of you.,positive,noon,60-70,Grenada,112523,340.0,331
27479,ed167662a5,But it was worth it ****.,But it was worth it ****.,positive,night,70-100,Guatemala,17915568,107160.0,167


# **Convert labels into 0 & 1**

In [3]:
train_df.rename(columns={"sentiment": "label"}, inplace=True)
train_df["label"] = train_df["label"].replace({"negative": 0, "positive": 1})

In [4]:
train_df = train_df.drop(train_df[train_df["label"] == "neutral"].index)
train_df

,textID,text,selected_text,label,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,0,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,0,night,31-45,Algeria,43851044,2381740.0,18
3,9642c003ef,what interview! leave me alone,leave me alone,0,morning,46-60,Andorra,77265,470.0,164
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",0,noon,60-70,Angola,32866272,1246700.0,26
6,6e0c6d75b1,2am feedings for the baby are fun when he is a...,fun,1,morning,0-20,Argentina,45195774,2736690.0,17
...,...,...,...,...,...,...,...,...,...,...
27475,b78ec00df5,enjoy ur night,enjoy,1,noon,21-30,Germany,83783942,348560.0,240
27476,4eac33d1c0,wish we could come see u on Denver husband l...,d lost,0,night,31-45,Ghana,31072940,227540.0,137
27477,4f4c4fc327,I`ve wondered about rake to. The client has ...,", don`t force",0,morning,46-60,Greece,10423054,128900.0,81
27478,f67aae2310,Yay good for both of you. Enjoy the break - y...,Yay good for both of you.,1,noon,60-70,Grenada,112523,340.0,331


# **Download some files in zip format & convert into unzip**

In [5]:
!wget http://downloads.cs.stanford.edu/nlp/data/glove.6B.zip

--2025-12-11 14:09:30--  http://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 64:ff9b::ab40:4016, 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|64:ff9b::ab40:4016|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip.1’

glove.6B.zip.1      100%[===================>] 822.24M  4.75MB/s    in 3m 17s  

2025-12-11 14:12:49 (4.17 MB/s) - ‘glove.6B.zip.1’ saved [862182613/862182613]



In [ ]:
!unzip glove.6B.zip

Archive:  glove.6B.zip
replace glove.6B.50d.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import numpy as np

words = dict()

def add_to_dict(d, filename):
  with open(filename, 'r') as f:
    for line in f.readlines():
      line = line.split(' ')

      try:
        d[line[0]] = np.array(line[1:], dtype=float)
      except:
        continue

add_to_dict(words, 'glove.6B.50d.txt')
words

In [ ]:
len(words)

# **nltk Download some Required Data or Files**

In [ ]:
import nltk

nltk.download('wordnet')

In [ ]:
nltk.download('omw-1.4')
nltk.download('wordnet')

In [ ]:
tokenizer = nltk.RegexpTokenizer(r"\w+")

tokenizer.tokenize('@user when a father is dysfunctional and is')

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def message_to_token_list(s):
  tokens = tokenizer.tokenize(s)
  lowercased_tokens = [t.lower() for t in tokens]
  lemmatized_tokens = [lemmatizer.lemmatize(t) for t in lowercased_tokens]
  useful_tokens = [t for t in lemmatized_tokens if t in words]

  return useful_tokens

message_to_token_list('@user feet a fathers is dysfunctional and is')

In [ ]:
def message_to_word_vectors(message, word_dict=words):
  processed_list_of_tokens = message_to_token_list(message)

  vectors = []

  for token in processed_list_of_tokens:
    if token not in word_dict:
      continue

    token_vector = word_dict[token]
    vectors.append(token_vector)

  return np.array(vectors, dtype=float)

message_to_word_vectors('@user when a father is dysfunctional and is').shape

# **Data Split for Training**

In [ ]:
train_df = train_df.sample(frac=1, random_state=1)
train_df.reset_index(drop=True, inplace=True)

split_index_1 = int(len(train_df) * 0.7)
split_index_2 = int(len(train_df) * 0.85)

train_df, val_df, test_df = train_df[:split_index_1], train_df[split_index_1:split_index_2], train_df[split_index_2:]

len(train_df), len(val_df), len(test_df)

In [ ]:
test_df

In [17]:
def df_to_X_y(dff):
  y = dff['label'].to_numpy().astype(int)

  all_word_vector_sequences = []

  for message in dff['text']:
    message_as_vector_seq = message_to_word_vectors(message)

    if message_as_vector_seq.shape[0] == 0:
      message_as_vector_seq = np.zeros(shape=(1, 50))

    all_word_vector_sequences.append(message_as_vector_seq)

  return all_word_vector_sequences, y

In [18]:
train_df['text'] = train_df['text'].fillna('')
train_df['text'] = train_df['text'].astype(str)

# Now create X, y
X_train, y_train = df_to_X_y(train_df)
print(len(X_train), len(X_train[0]))

11454 12


In [19]:
print(len(X_train), len(X_train[2]))

11454 12


In [1]:
sequence_lengths = []

for i in range(len(X_train)):
  sequence_lengths.append(len(X_train[i]))

import matplotlib.pyplot as plt

plt.title("Sequence Length graph")
plt.hist(sequence_lengths)

NameError: name 'X_train' is not defined

In [21]:
pd.Series(sequence_lengths).describe()

count    11454.000000
mean        13.369391
std          6.966197
min          1.000000
25%          8.000000
50%         12.000000
75%         19.000000
max         34.000000
dtype: float64

In [70]:
from copy import deepcopy

def pad_X(X, desired_sequence_length=57):
  X_copy = deepcopy(X)

  for i, x in enumerate(X):
    x_seq_len = x.shape[0]
    sequence_length_difference = desired_sequence_length - x_seq_len

    pad = np.zeros(shape=(sequence_length_difference, 50))
    X_copy[i] = np.concatenate([x, pad])

  return np.array(X_copy).astype(float)

In [71]:
X_train = pad_X(X_train)

X_train.shape

(11454, 57, 50)

In [72]:
y_train.shape

(11454,)

In [73]:
X_val, y_val = df_to_X_y(val_df)
X_val = pad_X(X_val)

X_val.shape, y_val.shape

((2454, 57, 50), (2454,))

In [74]:
X_test, y_test = df_to_X_y(test_df)
X_test = pad_X(X_test)

X_test.shape, y_test.shape

((2455, 57, 50), (2455,))

# **model work on LSTM**

In [75]:
#Our Model using LSTM
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

model = Sequential([])

model.add(layers.Input(shape=(57, 50)))
model.add(layers.LSTM(64, return_sequences=True))
model.add(layers.Dropout(0.2))
model.add(layers.LSTM(64, return_sequences=True))
model.add(layers.Dropout(0.2))
model.add(layers.LSTM(64, return_sequences=True))
model.add(layers.Dropout(0.2))
model.add(layers.Flatten())
model.add(layers.Dense(1, activation='sigmoid'))

# **It's summary of model**

In [76]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_3 (LSTM)                   │ (None, 57, 64)         │        29,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 57, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 57, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 57, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 57, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 57, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 3648)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │         3,649 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 99,137 (387.25 KB)

 Trainable params: 99,137 (387.25 KB)

 Non-trainable params: 0 (0.00 B)

In [77]:
#Saving our Model
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC
from tensorflow.keras.callbacks import ModelCheckpoint

cp = ModelCheckpoint('/content/tw.keras', save_best_only=True)
#Hyper-parameters of our model
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss=BinaryCrossentropy(),
              metrics=['accuracy', AUC(name='auc')])

In [78]:
frequencies = pd.value_counts(train_df['label'])
frequencies

/tmp/ipython-input-3931351302.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  frequencies = pd.value_counts(train_df['label'])


,count
label,
1,5971
0,5483


In [79]:
weights = {0: frequencies.sum() / frequencies[0], 1: frequencies.sum() / frequencies[1]}
weights

{0: np.float64(2.0890023709648005), 1: np.float64(1.9182716462904037)}

# **Train model through 49 epoches but you are use early stopping**

In [80]:
#Fitiing our training our model
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=49, callbacks=[cp], class_weight=weights)

Epoch 1/49
358/358 ━━━━━━━━━━━━━━━━━━━━ 9s 18ms/step - accuracy: 0.6030 - auc: 0.6407 - loss: 1.3011 - val_accuracy: 0.7620 - val_auc: 0.8422 - val_loss: 0.5002
Epoch 2/49
358/358 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.7686 - auc: 0.8462 - loss: 0.9727 - val_accuracy: 0.7722 - val_auc: 0.8629 - val_loss: 0.4751
Epoch 3/49
358/358 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.7873 - auc: 0.8654 - loss: 0.9166 - val_accuracy: 0.7869 - val_auc: 0.8738 - val_loss: 0.4525
Epoch 4/49
358/358 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.7913 - auc: 0.8734 - loss: 0.8920 - val_accuracy: 0.7967 - val_auc: 0.8810 - val_loss: 0.4362
Epoch 5/49
358/358 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.7962 - auc: 0.8788 - loss: 0.8717 - val_accuracy: 0.8011 - val_auc: 0.8865 - val_loss: 0.4242
Epoch 6/49
358/358 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.8123 - auc: 0.8932 - loss: 0.8242 - val_accuracy: 0.8093 - val_auc: 0.8911 - val_loss: 0.4179
Epoch 7/49
358/358 ━━━━━━━━━━━━━━━

In [81]:
from tensorflow.keras.models import load_model

best_model = load_model('/content/tw.keras')

In [82]:

test_predictions = (best_model.predict(X_test) > 0.5).astype(int)

from sklearn.metrics import classification_report

print(classification_report(y_test, test_predictions))

77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
              precision    recall  f1-score   support

           0       0.82      0.85      0.84      1154
           1       0.86      0.84      0.85      1301

    accuracy                           0.84      2455
   macro avg       0.84      0.84      0.84      2455
weighted avg       0.84      0.84      0.84      2455



# **Test**

In [85]:
a='Recession hit Veronique Branquinho, she has to quit her company, such a shame!'
b=message_to_token_list(a)
print(b)
c=message_to_word_vectors(a, word_dict=words)
print(c)
l=1
all_word_vector_sequences = []

if c.shape[0] == 0:
  c = np.zeros(shape=(1, 50))

all_word_vector_sequences.append(c)
d=pad_X(all_word_vector_sequences, desired_sequence_length=57)
print(best_model.predict(d))
t_predictions = (best_model.predict(d) > 0.5).astype(int)


print('The Sentiment of this tweet is: ',t_predictions)

['recession', 'hit', 'veronique', 'branquinho', 'she', 'ha', 'to', 'quit', 'her', 'company', 'such', 'a', 'shame']
[[ 1.8179e-01 -7.5381e-01  5.8120e-01 -1.0506e+00 -8.1497e-01  2.8252e-01
  -2.8160e-01 -1.6284e-02  7.5189e-02 -1.8392e-01  1.7294e-02 -3.3048e-01
  -7.8017e-01  4.7358e-01  1.0580e+00  8.4421e-01 -2.1532e-01 -8.3700e-01
  -1.7485e-01  3.1259e-01  8.7176e-02 -1.1799e+00 -4.8552e-01 -5.5888e-01
   5.1402e-01 -1.2151e+00 -5.0250e-01  4.7488e-01  8.5208e-01  1.7429e+00
   2.3028e+00  5.0215e-01  1.2699e+00  1.5825e-01 -9.3648e-01 -5.8294e-01
  -1.1013e+00 -3.2265e-01  5.3048e-01 -1.3961e+00 -1.9727e+00 -2.2009e-01
   4.7897e-01 -2.6444e-01  2.0521e-01  2.7641e-01  1.0516e+00  4.2095e-01
   3.4474e-01 -2.7933e-02]
 [-4.1659e-01 -4.7596e-01  9.5744e-01  2.7019e-01  1.7657e-01  2.4828e-01
  -1.2987e+00  5.3851e-01  3.5336e-01  5.8221e-01 -3.3079e-01 -5.9680e-01
  -9.7055e-01  7.2084e-01  4.9463e-01 -8.3398e-01  1.2236e-01 -3.7237e-01
  -1.4546e+00  4.1384e-01 -3.6311e-01  2.202

In [86]:
#Use this function if you want remove unneeded symbols
import re

def text_cleaning(text):
  text = re.sub(r'@[A-Za-z0-9]+', '', text)     # removing @mentions
  text = re.sub(r'@[A-Za-zA-Z0-9]+', '', text)  # removing @mentions
  text = re.sub(r'@[A-Za-z]+', '', text)        # removing @mentions
  text = re.sub(r'@[-)]+', '', text)            # removing @mentions
  text = re.sub(r'#', '', text )                # removing '#' sign
  text = re.sub(r'RT[\s]+', '', text)           # removing RT
  text = re.sub(r'https?\/\/\S+', '', text)     # removing the hyper link
  text = re.sub(r'&[a-z;]+', '', text)          # removing '>'

  return text

# applying the text cleaning function on tweets
train_df['selected_text'] = train_df['selected_text'].apply(text_cleaning)
train_df.head(10)

,textID,text,selected_text,label,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,733d30d2f7,Haha. It`s pretty good they`re making somethi...,good,1,morning,0-20,Ukraine,43733762,579320.0,75
1,42e9dce94a,Happy Mama`s day to all mothers,Happy,1,noon,60-70,Zambia,18383955,743390.0,25
2,c5990861b8,I`m sure you`d consider it if they offerred t...,I`m sure you`d consider,1,noon,21-30,Azerbaijan,10139177,82658.0,123
3,f9483e7dc1,Sometimes the things you say hurt the ones you...,hurt,0,noon,21-30,Côte d'Ivoire,26378274,318000.0,83
4,3d0896ec67,you can put a saucepan full of water on the c...,...indian style scrabbled are the best!,1,morning,46-60,Rwanda,12952218,24670.0,525
5,fcdbdfce85,Thanks madam.. you`re lucky coz you had a won...,you had a wonderful dog.. and sooo cute...,1,night,31-45,Netherlands,17134872,33720.0,508
6,7caac2808c,Happy Star Wars Day!!!,Happy Star Wars Day!!!,1,morning,46-60,South Korea,51269185,97230.0,527
7,5.20E+45,"can`t wait to see `Transformers 2`.. C`me on, ...",Yippiee!,1,night,70-100,Barbados,287375,430.0,668
8,bfcac206a0,ok nevermind. photo was set to private. sorry.,sorry.,0,night,70-100,Slovenia,2078938,20140.0,103
9,ddbdb89b1b,Going to bed. Goodnight! x,Going to bed. Goodnight! x,1,morning,0-20,Ukraine,43733762,579320.0,75
